# 01 – Coleta das checagens da Agência Lupa

**Entrada:** API da Agência Lupa (`api.lupa.news`)  
**Saída:** `data/raw/checagens_lupa_raw.csv` (coleta original, usada no TCC)

- **Célula de coleta (opcional):** refaz a consulta à API. Como a API muda com o tempo, o resultado é salvo em `data/raw/checagens_lupa_recoleta.csv` e **não substitui** a base original.
- **Célula de conferência:** lê a base original e reproduz os totais do TCC (1.191 brutas; 984 em 2020–2022).

Para reproduzir o TCC, **basta executar a célula de conferência**.

In [ ]:
# 1. Importar bibliotecas essenciais 
import requests         
# Requisições HTTP para acessar a API da Lupa 
import time              
import csv               
# Controle de tempo entre as requisições 
# Exportação dos dados para arquivo CSV 

# 2. Configurar URL e cabeçalhos da API da Lupa 
URL = "https://api.lupa.news/api/v1/search" 
HEADERS = { 
    "Content-Type": "application/json", 
    "Origin": "https://lupa.uol.com.br", 
    "Referer": "https://lupa.uol.com.br/", 
    "User-Agent": "Mozilla/5.0" 
} 

# 3. Definir palavras-chave de interesse 
palavras_chave = [ 
    "vacina", "antivacina", "vacinação obrigatória", 
    "cloroquina", "ivermectina", "tratamento precoce", 
    "chip vacina", "bill gates", "5G", "nova ordem mundial", 
    "COVID é mentira", "coronavírus não existe", "morte inventada", 
    "STF covid", "OMS", "médico denuncia", "Doria covid", "Bolsonaro covid", 
    "fake news", "desinformação", "boato", "notícia falsa" 
]
 
# 4. Definir parâmetros de paginação da API 
LIMITE_PAGINAS = 50       
# Número máximo de páginas por termo 
POR_PAGINA = 10           
# Resultados por página (offset) 

# 5. Lista principal para armazenar os dados coletados 
todos_resultados = [] 

# 6. Função para buscar dados com base em uma palavra-chave 
def buscar_lupa(palavra): 
    resultados = [] 
    for pagina in range(LIMITE_PAGINAS): 
        offset = pagina * POR_PAGINA 
        payload = { 
            "query": palavra, 
            "tags": ["Falso"], 
            "sort": ["published_at", "desc"], 
            "types": [], 
            "categories": [], 
            "startDate": "2020-01-01T00:00:01.000Z", 
            "endDate": "2023-12-31T23:59:59.999Z", 
            "from": offset 
        } 
        response = requests.post(URL, headers=HEADERS, json=payload) 
        if response.status_code != 200:
            # A string f agora está toda na mesma linha
            print(f"Página {pagina+1} da palavra-chave '{palavra}' falhou. Código: {response.status_code}")
            break

        data = response.json() 
        hits = data.get("hits", []) 

        if not hits: 
            print(f"Fim dos resultados para '{palavra}'.") 
            break

        for item in hits: 
            resultados.append({ 
                "titulo": item.get("name"), 
                "url": "https://lupa.uol.com.br/" + item.get("full_slug"), 
                "data_publicacao": item.get("published_at") 
            }) 
        print(f"Página {pagina+1} de '{palavra}' finalizada com sucesso.") 
        time.sleep(1)  # Evita sobrecarga no servidor 
    return resultados

# 7. Executar busca para cada palavra-chave e acumular resultados 
for palavra in palavras_chave: 
    print(f"\nIniciando busca por '{palavra}'...") 
    resultados = buscar_lupa(palavra) 
    todos_resultados.extend(resultados) 

# 8. Eliminar duplicatas com base na URL 
unicos = {item['url']: item for item in todos_resultados} 
lista_final = list(unicos.values())

# 9. Exportar resultados finais para arquivo CSV 
# ATENÇÃO: a base usada no TCC é '../data/raw/checagens_lupa_raw.csv' (coleta original).
# Uma nova execução consulta a API hoje e pode retornar resultados diferentes;
# por isso a recoleta é salva em outro arquivo, sem sobrescrever a base original.
output_path = '../data/raw/checagens_lupa_recoleta.csv' 
with open(output_path, mode='w', newline='', encoding='utf-8') as csvfile: 
    campos = ["titulo", "url", "data_publicacao"] 
    writer = csv.DictWriter(csvfile, fieldnames=campos) 
    writer.writeheader() 
    writer.writerows(lista_final) 

# 10. Mensagem final de conclusão 
print(f"\nBUSCA CONCLUÍDA: {len(lista_final)} registros únicos salvos em '{output_path}'") 


Iniciando busca por 'vacina'...
Página 1 de 'vacina' finalizada com sucesso.
Página 2 de 'vacina' finalizada com sucesso.
Página 3 de 'vacina' finalizada com sucesso.
Página 4 de 'vacina' finalizada com sucesso.
Página 5 de 'vacina' finalizada com sucesso.
Página 6 de 'vacina' finalizada com sucesso.
Página 7 de 'vacina' finalizada com sucesso.
Página 8 de 'vacina' finalizada com sucesso.
Página 9 de 'vacina' finalizada com sucesso.
Página 10 de 'vacina' finalizada com sucesso.
Página 11 de 'vacina' finalizada com sucesso.
Página 12 de 'vacina' finalizada com sucesso.
Página 13 de 'vacina' finalizada com sucesso.
Página 14 de 'vacina' finalizada com sucesso.
Página 15 de 'vacina' finalizada com sucesso.
Página 16 de 'vacina' finalizada com sucesso.
Página 17 de 'vacina' finalizada com sucesso.
Página 18 de 'vacina' finalizada com sucesso.
Página 19 de 'vacina' finalizada com sucesso.
Página 20 de 'vacina' finalizada com sucesso.
Página 21 de 'vacina' finalizada com sucesso.
Página 22 

SSLError: HTTPSConnectionPool(host='api.lupa.news', port=443): Max retries exceeded with url: /api/v1/search (Caused by SSLError(SSLEOFError(8, 'EOF occurred in violation of protocol (_ssl.c:992)')))

In [1]:
import pandas as pd

# Caminho do seu arquivo da Lupa
caminho_lupa = '../data/raw/checagens_lupa_raw.csv'  # base original usada no TCC

# Ler o arquivo
df_lupa = pd.read_csv(caminho_lupa)

print("1. Total Bruto Lupa (22 palavras-chave + Selo Falso - 2020 a 2023):", len(df_lupa))

# Converter a coluna de data para o formato do Python
df_lupa['data_publicacao'] = pd.to_datetime(df_lupa['data_publicacao'], errors='coerce')

# Filtrar apenas o período do TCC (2020 a 2022)
df_lupa_final = df_lupa[(df_lupa['data_publicacao'] >= '2020-01-01') & (df_lupa['data_publicacao'] <= '2022-12-31')]

print("2. Amostra Final Lupa (Apenas 2020 a 2022):", len(df_lupa_final))

1. Total Bruto Lupa (22 palavras-chave + Selo Falso - 2020 a 2023): 1191
2. Amostra Final Lupa (Apenas 2020 a 2022): 984
